In [24]:
import os
import pandas as pd
import numpy as np
from numba import njit, float64, int64, uint64,types
from numba.typed import Dict

In [25]:

# 현재 파일들이 있는 그 위치 그대로 설정
Base_dir = "C:/Users/user/Desktop/IDS_masters/9) Car-Hacking Dataset"

# 파일 이름에 포함된 단어로 공격 유형 구분
attack_mapping = {
    "Dos": 1,
    "Fuzzing": 2,
    "Spoofing":4
}

attack_files = []

# 폴더 안을 바로 검사
for attack_name, attack_id in attack_mapping.items():
    attack_dir = os.path.join(Base_dir, attack_name)
    if not os.path.isdir(attack_dir):
        continue

    for fname in os.listdir(attack_dir):
        if fname.endswith(".csv"):
            attack_files.append({
                "path": os.path.join(attack_dir, fname),
                "attack_id": attack_id
            })

In [26]:
#################################
# 2. Visualization Mirgu Dataset
#################################
hash_cache = {}

# [ADD] payload 8바이트 리스트로 만드는 함수 (너가 쓰던 스타일)
def parse_payload(row):
    # row에는 b0~b7 컬럼이 있고, 이미 0패딩되어 있음
    return [int(row[f"b{i}"]) for i in range(8)]

def process_csv_file(path, attack_id):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            if len(parts) < 4:
                continue

            ts_str, canid_raw, dlc_str = parts[0], parts[1], parts[2]
            label = parts[-1].strip()         # [MINOR] strip
            data_tokens = parts[3:-1]

            # dlc/ts 파싱
            try:
                ts = float(ts_str)
                dlc = int(dlc_str)
            except:
                continue

            # payload bytes: DLC 만큼만 읽고, 8바이트로 0 패딩
            payload = []
            for i in range(min(dlc, len(data_tokens), 8)):
                tok = data_tokens[i].strip()
                if tok == "" or tok.lower() == "nan":
                    payload.append(0)
                else:
                    try:
                        payload.append(int(tok, 16))
                    except:
                        payload.append(0)

            payload += [0] * (8 - len(payload))
            payload = payload[:8]

            rows.append([ts, canid_raw, dlc, *payload, label])

    df = pd.DataFrame(
        rows,
        columns=["timestamp", "CAN_ID", "DLC"] + [f"b{i}" for i in range(8)] + ["Label"]
    )

    # CAN_ID int 변환
    df["int_CAN_ID"] = df["CAN_ID"].apply(lambda x: int(str(x).strip(), 16)).astype(np.int64)


    # Payloads 컬럼 추가 
    df["Payloads"] = df.apply(parse_payload, axis=1).tolist()

    # 라벨링
    df["Labeling"] = df["Label"].map({"T": attack_id, "R": 0}).fillna(0).astype(int)

    df = df[["timestamp","int_CAN_ID","Payloads","Labeling"]]

    return df


In [27]:
# =========================================================
# 1. 설정 (Testing: BMW)
# =========================================================
# [경로 수정] 본인의 BMW 데이터 경로로 수정하세요
BASE_DIR = "C:/Users/user/Desktop/IDS_masters/9) Car-Hacking Dataset"
OUTPUT_DIR = "C:/Users/user/Desktop/IDS_masters/features"

# 기존 스케일러는 30개 피처용이므로 15개 피처에는 사용할 수 없어 주석 처리했습니다.
# SCALER_PATH = "C:/Users/user/Desktop/IDS_masters/features/robust_scaler_30feat.pkl" 

ATTACK_DIR = {"Dos": 1, "Fuzzing": 2, "Spoofing": 4}

WINDOW_SIZE, STRIDE = 128, 64

# 요청하신 15개 피처 이름
FEATURE_NAMES_15 = [
    "IAT", "Is_Zero", "Payload_Ent", "Complexity", 
    "Ham_Rate", "Freq", "Continuity", "Diff_Ent", "ID_Ent",
    "Freq_Fast_Z", "Jit_Fast_Z","?"
]

# =========================================================
# 2. Numba 엔진 (15개 피처 로직 이식)
# =========================================================
@njit
def popcount64(x):
    c = 0
    v = int64(x)
    while v:
        v &= v - int64(1)
        c += 1
    return c

@njit
def pack_payload_u64(row):
    v = uint64(0)
    for i in range(8):
        v |= uint64(row[i]) << (i * 8)
    return v

@njit
def update_ema_z(val, cid, ema_map, sq_ema_map, alpha):
    """
    EMA 기반 Z-Score 계산 (신규 로직)
    """
    if cid not in ema_map:
        ema_map[cid] = float64(val)
        sq_ema_map[cid] = float64(val ** 2)
        return 0.0

    mean = ema_map[cid]
    sq_mean = sq_ema_map[cid]
    
    var = sq_mean - (mean ** 2)
    if var < 0: var = 0.0
    std = np.sqrt(var)

    z = 0.0
    if std > 1e-9:
        z = (val - mean) / std
        if z > 5.0: z = 5.0
        elif z < -5.0: z = -5.0

    ema_map[cid] = (1.0 - alpha) * mean + alpha * val
    sq_ema_map[cid] = (1.0 - alpha) * sq_mean + alpha * (val ** 2)
    
    return z

@njit(fastmath=True)
def calculate_features_15_numba(timestamps, can_ids, payloads):
    # dlcs는 서명 호환성을 위해 받지만 내부 계산에는 사용하지 않음
    n = len(timestamps)
    features = np.zeros((n, 12), dtype=np.float64)
    
    # --- [기존 변수들] ---
    last_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)
    last_id_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    id_ham_ema = Dict.empty(key_type=types.int64, value_type=types.float64)
    alpha_ham = 0.05
    eps = 1e-9

    # --- [신규 변수들] ---
    last_freq_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_jit_map_val = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_ent_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    # Z-Score용 EMA 맵 (Alpha=0.1 Fast)
    ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)

    G_KEY = np.int64(-1)
    # ID별 (Local) 정규화용
    ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    # 전체 네트워크 (Global) 정규화용 - 여기서 정의합니다!
    ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    prev_global_time = timestamps[0]

    for i in range(n):
        # 64개마다 윈도우 빈도 초기화
        if (i % 64) == 0:
            last_id_map.clear()
            
        ts = timestamps[i]
        cid = can_ids[i]
        row = payloads[i]
        
        if np.isnan(ts): ts = prev_global_time

        # # 1. New ID 감지 (가장 먼저 수행)
        # is_new_id = 0.0 if cid in last_time_map else 1.0
        
        # --- [공통 물리량 계산] ---
        # 1. IAT
        curr_iat = 0.0
        if cid in last_time_map:
            curr_iat = max(0.0, ts - last_time_map[cid])
        else:
            curr_iat = 0.001
        
        # 2. Frequency (1/IAT)
        curr_freq = 0.0
        if curr_iat > 1e-9:
            curr_freq = 1.0 / curr_iat
            
        # 3. Jitter (|Current IAT - Last IAT|)
        curr_jit = 0.0
        if cid in last_iat_map:
            curr_jit = np.abs(curr_iat - last_iat_map[cid])
        last_iat_map[cid] = curr_iat
        
        # --- [Part 1] 기존 9개 피처 ---
        cur_bytes = pack_payload_u64(row)
        rel_change = 0.0
        if cid in last_payload_map:
            diff = cur_bytes ^ last_payload_map[cid]
            h_dist = float64(popcount64(diff))
            if cid in id_ham_ema:
                avg_h = id_ham_ema[cid]
                rel_change = h_dist / (avg_h + 0.1) 
                id_ham_ema[cid] = (1.0 - alpha_ham) * avg_h + alpha_ham * h_dist
            else:
                rel_change = 1.0
                id_ham_ema[cid] = h_dist
        else:
            rel_change = 0.0
        last_payload_map[cid] = cur_bytes

        p_counts = np.zeros(256, dtype=np.int64)
        for b in row: p_counts[b] += 1
        ent = 0.0
        for c in p_counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)

        features[i, 0] = np.log1p(curr_iat * 1000.0) / 7.0  # 1. IAT
        features[i, 1] = 1.0 if cid == 0 else 0.0           # 2. Is_Zero
        features[i, 2] = ent / 2.1                          # 3. Payload_Ent
        features[i, 3] = np.log1p(ent * rel_change)         # 4. Complexity
        features[i, 4] = np.log1p(rel_change / (curr_iat + eps)) / 10.0 # 5. Ham_Rate
        
        cnt = last_id_map.get(cid, 0.0) + 1.0
        last_id_map[cid] = cnt
        features[i, 5] = cnt / 128.0                        # 6. Freq (Local)

        features[i, 6] = np.log1p(rel_change) / 5.0         # 7. Continuity

        diffs = np.zeros(7, dtype=np.int64)
        for b_idx in range(7):
            diffs[b_idx] = (int64(row[b_idx+1]) - int64(row[b_idx])) % 256
        d_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
        for d in diffs: d_counts[d] = d_counts.get(d, 0.0) + 1.0
        d_ent = 0.0
        for dv in d_counts:
            p = d_counts[dv] / 7.0
            d_ent -= p * np.log(p + 1e-9)
        features[i, 7] = d_ent / 1.94                          # 8. Diff_Ent

        if i >= 127:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for j in range(i-127, i+1):
                wid = can_ids[j]
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            wi_ent = 0.0
            for k_id in win_id_counts:
                pk = win_id_counts[k_id] / 128.0
                wi_ent -= pk * np.log(pk + 1e-9)
            features[i, 8] = wi_ent / 4.85                      # 9. ID_Ent
        else:
            features[i, 8] = 0.0

        # --- [Part 2] 신규 6개 피처 ---
        
        # # 10. Freq_Slope
        # if cid in last_freq_map:
        #     features[i, 9] = curr_freq - last_freq_map[cid]
        # else:
        #     features[i, 9] = 0.0
        # last_freq_map[cid] = curr_freq

        # # 11. Jit_Slope
        # if cid in last_jit_map_val:
        #     features[i, 10] = curr_jit - last_jit_map_val[cid]
        # else:
        #     features[i, 10] = 0.0
        # last_jit_map_val[cid] = curr_jit

        # # 12. Ent_Slope
        # if cid in last_ent_map:
        #     features[i, 11] = ent - last_ent_map[cid]
        # else:
        #     features[i, 11] = 0.0
        # last_ent_map[cid] = ent

        # 13. Freq_Fast_Z
        features[i, 9] = update_ema_z(curr_freq, cid, ema_freq, sq_ema_freq, 0.001)

        # 14. Jit_Fast_Z
        features[i, 10] = update_ema_z(curr_jit, cid, ema_jit, sq_ema_jit, 0.001)
        # (추가 제안) 16번 피쳐: 전체 네트워크 빈도 정규화 (신규)
        # 모든 ID가 들어올 때마다 '-1'이라는 동일한 키로 업데이트합니다.
        features[i, 11] = update_ema_z(curr_freq, G_KEY, ema_global, sq_ema_global, 0.001)

        

        # # 15. Bus_Load
        # global_iat = ts - prev_global_time if i > 0 else 0.001
        # features[i, 12] = 1.0 / (global_iat + 1e-9)
        
        prev_global_time = ts
        
        # 상태 업데이트
        last_time_map[cid] = ts

    return features


In [28]:
# ==========================================
# 4. Making Feature with Numba
# ==========================================

def Make_feature(path, attack_id):

    df = process_csv_file(path, attack_id)

    # ======== to numpy ========== #
    timestamps = df["timestamp"].to_numpy(np.float32)
    can_ids = df["int_CAN_ID"].to_numpy(np.int64)
    payloads = np.array(df["Payloads"].tolist(), dtype=np.uint8)
    labels = df["Labeling"].to_numpy(np.int64)

    
    # ======== calculate feature ========== #
    feature9 = calculate_features_15_numba(timestamps, can_ids, payloads)
    print(feature9.shape)

    return feature9, labels

In [29]:
# ==========================================
# 5. Slide Window and Label
# ==========================================
def Sliding_Window_and_Labeling(feature, label, win_size=128, stride=64):
    windows = []
    labels = []
    n = feature.shape[0]
    for start in range(0, n-win_size+1 , stride):
        end = start + win_size
        windows.append(feature[start:end])
        labels.append(label[start:end])


    return (
        np.stack(windows, axis=0).astype(np.float32),
        np.stack(labels, axis=0).astype(np.int64)
    )

In [30]:
# ==========================================
# 6. main
# ==========================================
all_x = []
all_y = []

for item in attack_files:
    feature9, labels = Make_feature(item["path"], item["attack_id"]) # 각 feature 추출
    windows, y = Sliding_Window_and_Labeling(feature9,labels) # 윈도우 만들기

    all_x.append(windows)
    all_y.append(y)

all_x_win = np.concatenate(all_x, axis=0)
all_y_win = np.concatenate(all_y, axis=0)

(3665771, 12)
(3838860, 12)
(4443142, 12)
(4621702, 12)


In [31]:
# ==========================================
# 7. Save
# ==========================================
import numpy as np
np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_0212_816.npz",
    X = all_x_win.astype(np.float32),
    y = all_y_win.astype(np.int64)
    )

print(f" Saved dataset")

print("X shape:", all_x_win.shape)
print("y shape:", all_y_win.shape)

 Saved dataset
X shape: (258893, 128, 12)
y shape: (258893, 128)
